In [1]:
!nvidia-smi
!pip -q install -U ultralytics pyyaml

Tue Mar 17 10:14:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   34C    P0             53W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
from pathlib import Path


DRIVE_DATASET_ROOT = Path("/content/drive/MyDrive/Chi_Tam/VR-TSD.v2i.coco")


LOCAL_YOLO_ROOT = Path("/content/VR-TSD_YOLO")


SAVE_ROOT = Path("/content/drive/MyDrive/Chi_Tam/output")
SAVE_ROOT.mkdir(parents=True, exist_ok=True)

print("DRIVE_DATASET_ROOT =", DRIVE_DATASET_ROOT)
print("LOCAL_YOLO_ROOT    =", LOCAL_YOLO_ROOT)
print("SAVE_ROOT          =", SAVE_ROOT)
assert DRIVE_DATASET_ROOT.exists(), f"Không thấy dataset tại: {DRIVE_DATASET_ROOT}"

DRIVE_DATASET_ROOT = /content/drive/MyDrive/Chi_Tam/VR-TSD.v2i.coco
LOCAL_YOLO_ROOT    = /content/VR-TSD_YOLO
SAVE_ROOT          = /content/drive/MyDrive/Chi_Tam/output


In [4]:
import os
import json
import shutil
from collections import defaultdict

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def find_json_file(split_dir: Path):
    cands = list(split_dir.glob("*.json"))
    if len(cands) == 0:
        raise FileNotFoundError(f"Không thấy file json trong {split_dir}")
    # ưu tiên file có chữ coco / annotation
    cands = sorted(cands, key=lambda p: (("coco" not in p.name.lower()) and ("annot" not in p.name.lower()), p.name))
    return cands[0]

def find_image_file(split_dir: Path, file_name: str):
    p = split_dir / file_name
    if p.exists():
        return p

    p2 = split_dir / Path(file_name).name
    if p2.exists():
        return p2

    target = Path(file_name).name.lower()
    matches = [x for x in split_dir.rglob("*") if x.is_file() and x.suffix.lower() in IMG_EXTS and x.name.lower() == target]
    if matches:
        return matches[0]

    return None

def coco_to_yolo_bbox(bbox, w, h):
    x, y, bw, bh = bbox
    xc = (x + bw / 2.0) / w
    yc = (y + bh / 2.0) / h
    bw = bw / w
    bh = bh / h
    return xc, yc, bw, bh

def prepare_split(split_name: str, split_dir: Path, out_root: Path, cat_id_to_idx: dict):
    out_img_dir = out_root / split_name / "images"
    out_lbl_dir = out_root / split_name / "labels"
    out_img_dir.mkdir(parents=True, exist_ok=True)
    out_lbl_dir.mkdir(parents=True, exist_ok=True)

    ann_path = find_json_file(split_dir)
    print(f"[{split_name}] annotation:", ann_path)

    with open(ann_path, "r", encoding="utf-8") as f:
        coco = json.load(f)

    images = {img["id"]: img for img in coco["images"]}
    anns_by_img = defaultdict(list)
    for ann in coco["annotations"]:
        if ann.get("iscrowd", 0) == 1:
            continue
        anns_by_img[ann["image_id"]].append(ann)

    n_copy = 0
    n_labels = 0

    for image_id, img_info in images.items():
        file_name = img_info["file_name"]
        w = float(img_info["width"])
        h = float(img_info["height"])

        src_img = find_image_file(split_dir, file_name)
        if src_img is None:
            raise FileNotFoundError(f"Không tìm thấy ảnh {file_name} trong {split_dir}")

        dst_img = out_img_dir / src_img.name
        if not dst_img.exists():
            shutil.copy2(src_img, dst_img)
            n_copy += 1

        label_path = out_lbl_dir / (dst_img.stem + ".txt")
        lines = []

        for ann in anns_by_img.get(image_id, []):
            cat_id = ann["category_id"]
            cls = cat_id_to_idx[cat_id]
            xc, yc, bw, bh = coco_to_yolo_bbox(ann["bbox"], w, h)

            # clip an toàn
            xc = min(max(xc, 0.0), 1.0)
            yc = min(max(yc, 0.0), 1.0)
            bw = min(max(bw, 1e-8), 1.0)
            bh = min(max(bh, 1e-8), 1.0)

            lines.append(f"{cls} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}")

        with open(label_path, "w", encoding="utf-8") as f:
            f.write("\n".join(lines))

        n_labels += 1

    print(f"[{split_name}] copied images: {n_copy}, labels: {n_labels}")

def convert_drive_coco_to_local_yolo(drive_root: Path, local_root: Path):
    if local_root.exists():
        shutil.rmtree(local_root)
    local_root.mkdir(parents=True, exist_ok=True)

    # split names thường là train/valid/test
    split_dirs = {}
    for name in ["train", "valid", "test", "val"]:
        d = drive_root / name
        if d.exists() and d.is_dir():
            split_dirs[name] = d

    assert "train" in split_dirs, "Dataset thiếu thư mục train"
    assert ("valid" in split_dirs or "val" in split_dirs), "Dataset thiếu thư mục valid/val"

    val_key = "valid" if "valid" in split_dirs else "val"

    # đọc categories từ train json
    train_json = find_json_file(split_dirs["train"])
    with open(train_json, "r", encoding="utf-8") as f:
        train_coco = json.load(f)

    cats = sorted(train_coco["categories"], key=lambda x: x["id"])
    names = [c["name"] for c in cats]
    cat_id_to_idx = {c["id"]: i for i, c in enumerate(cats)}

    # convert từng split
    prepare_split("train", split_dirs["train"], local_root, cat_id_to_idx)
    prepare_split("valid", split_dirs[val_key], local_root, cat_id_to_idx)

    if "test" in split_dirs:
        prepare_split("test", split_dirs["test"], local_root, cat_id_to_idx)

    # tạo data.yaml
    data_yaml = local_root / "data.yaml"
    data_obj = {
        "path": str(local_root),
        "train": "train/images",
        "val": "valid/images",
        "test": "test/images" if (local_root / "test" / "images").exists() else "valid/images",
        "names": names,
    }

    import yaml
    with open(data_yaml, "w", encoding="utf-8") as f:
        yaml.safe_dump(data_obj, f, sort_keys=False, allow_unicode=True)

    return data_yaml, names

DATA_YAML, NAMES = convert_drive_coco_to_local_yolo(DRIVE_DATASET_ROOT, LOCAL_YOLO_ROOT)
NC = len(NAMES)

print("DATA_YAML =", DATA_YAML)
print("NC =", NC)
print("5 class đầu:", NAMES[:5])

[train] annotation: /content/drive/MyDrive/Chi_Tam/VR-TSD.v2i.coco/train/_annotations.coco.json
[train] copied images: 5668, labels: 5668
[valid] annotation: /content/drive/MyDrive/Chi_Tam/VR-TSD.v2i.coco/valid/_annotations.coco.json
[valid] copied images: 1259, labels: 1259
[test] annotation: /content/drive/MyDrive/Chi_Tam/VR-TSD.v2i.coco/test/_annotations.coco.json
[test] copied images: 1151, labels: 1151
DATA_YAML = /content/VR-TSD_YOLO/data.yaml
NC = 59
5 class đầu: ['objects-G3Zw-orpg-traffic-sign-1TU9-U4is', '102-cam-di-nguoc-chieu', '103a-cam-oto', '103b-cam-oto-re-phai', '103c-cam-oto-re-trai']


In [5]:
import math
import torch
import torch.nn as nn

class ConvBNAct(nn.Module):
    def __init__(self, c1, c2, k=1, s=1, g=1, act=True):
        super().__init__()
        p = k // 2
        self.conv = nn.Conv2d(c1, c2, k, s, p, groups=g, bias=False)
        self.bn = nn.BatchNorm2d(c2)
        self.act = nn.SiLU(inplace=True) if act else nn.Identity()

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))

class SqueezeExcite(nn.Module):
    def __init__(self, c, r=0.25):
        super().__init__()
        hidden = max(8, int(c * r))
        self.avg = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(c, hidden, 1, bias=True),
            nn.SiLU(inplace=True),
            nn.Conv2d(hidden, c, 1, bias=True),
            nn.Sigmoid()
        )

    def forward(self, x):
        w = self.fc(self.avg(x))
        return x * w

class UIB(nn.Module):
    # Universal Inverted Bottleneck kiểu MobileNet-like
    def __init__(self, c1, c2, k=3, s=1, e=6.0):
        super().__init__()
        hidden = max(16, int(round(c1 * e)))
        self.use_res = (s == 1 and c1 == c2)

        self.block = nn.Sequential(
            ConvBNAct(c1, hidden, k=1, s=1, act=True),
            ConvBNAct(hidden, hidden, k=k, s=s, g=hidden, act=True),
            SqueezeExcite(hidden, r=0.25),
            ConvBNAct(hidden, c2, k=1, s=1, act=False),
        )

    def forward(self, x):
        y = self.block(x)
        return x + y if self.use_res else y

class UIBDown(nn.Module):
    def __init__(self, c1, c2, k=3, s=2, e=6.0):
        super().__init__()
        hidden = max(16, int(round(c1 * e)))

        self.block = nn.Sequential(
            ConvBNAct(c1, hidden, k=1, s=1, act=True),
            ConvBNAct(hidden, hidden, k=k, s=s, g=hidden, act=True),
            SqueezeExcite(hidden, r=0.25),
            ConvBNAct(hidden, c2, k=1, s=1, act=False),
        )

    def forward(self, x):
        return self.block(x)

In [6]:
import math
import torch
import torch.nn as nn

class ConvBNAct(nn.Module):
    def __init__(self, c1, c2, k=1, s=1, g=1, act=True):
        super().__init__()
        p = k // 2
        self.conv = nn.Conv2d(c1, c2, k, s, p, groups=g, bias=False)
        self.bn = nn.BatchNorm2d(c2)
        self.act = nn.SiLU(inplace=True) if act else nn.Identity()

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))

class SqueezeExcite(nn.Module):
    def __init__(self, c, r=0.25):
        super().__init__()
        hidden = max(8, int(c * r))
        self.avg = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(c, hidden, 1, bias=True),
            nn.SiLU(inplace=True),
            nn.Conv2d(hidden, c, 1, bias=True),
            nn.Sigmoid()
        )

    def forward(self, x):
        w = self.fc(self.avg(x))
        return x * w

class UIB(nn.Module):
    # Universal Inverted Bottleneck kiểu MobileNet-like
    def __init__(self, c1, c2, k=3, s=1, e=6.0):
        super().__init__()
        hidden = max(16, int(round(c1 * e)))
        self.use_res = (s == 1 and c1 == c2)

        self.block = nn.Sequential(
            ConvBNAct(c1, hidden, k=1, s=1, act=True),
            ConvBNAct(hidden, hidden, k=k, s=s, g=hidden, act=True),
            SqueezeExcite(hidden, r=0.25),
            ConvBNAct(hidden, c2, k=1, s=1, act=False),
        )

    def forward(self, x):
        y = self.block(x)
        return x + y if self.use_res else y

class UIBDown(nn.Module):
    def __init__(self, c1, c2, k=3, s=2, e=6.0):
        super().__init__()
        hidden = max(16, int(round(c1 * e)))

        self.block = nn.Sequential(
            ConvBNAct(c1, hidden, k=1, s=1, act=True),
            ConvBNAct(hidden, hidden, k=k, s=s, g=hidden, act=True),
            SqueezeExcite(hidden, r=0.25),
            ConvBNAct(hidden, c2, k=1, s=1, act=False),
        )

    def forward(self, x):
        return self.block(x)

In [7]:
MODEL_YAML = LOCAL_YOLO_ROOT / "yolo11_uib_p2_traffic.yaml"

model_yaml_text = f"""
nc: {NC}

backbone:
  - [-1, 1, UIBDown, [32, 3, 2, 6.0]]
  - [-1, 1, UIBDown, [64, 3, 2, 6.0]]
  - [-1, 1, UIB,     [64, 3, 1, 6.0]]
  - [-1, 1, UIB,     [64, 3, 1, 6.0]]

  - [-1, 1, UIBDown, [128, 3, 2, 6.0]]
  - [-1, 1, UIB,     [128, 3, 1, 6.0]]
  - [-1, 1, UIB,     [128, 3, 1, 6.0]]

  - [-1, 1, UIBDown, [256, 3, 2, 6.0]]
  - [-1, 1, UIB,     [256, 3, 1, 6.0]]
  - [-1, 1, UIB,     [256, 3, 1, 6.0]]

  - [-1, 1, UIBDown, [512, 3, 2, 6.0]]
  - [-1, 1, UIB,     [512, 3, 1, 6.0]]
  - [-1, 1, UIB,     [512, 3, 1, 6.0]]
  - [-1, 1, SPPF,    [512, 5]]
  - [-1, 2, C2PSA,   [512]]

head:
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 9], 1, Concat, [1]]
  - [-1, 2, C3k2, [256, False]]

  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 6], 1, Concat, [1]]
  - [-1, 2, C3k2, [128, False]]

  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 3], 1, Concat, [1]]
  - [-1, 2, C3k2, [64, False]]

  - [-1, 1, Conv, [64, 3, 2]]
  - [[-1, 20], 1, Concat, [1]]
  - [-1, 2, C3k2, [128, False]]

  - [-1, 1, Conv, [128, 3, 2]]
  - [[-1, 17], 1, Concat, [1]]
  - [-1, 2, C3k2, [256, False]]

  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 14], 1, Concat, [1]]
  - [-1, 2, C3k2, [512, True]]

  - [[23, 26, 29, 32], 1, Detect, [nc]]
""".strip()

MODEL_YAML.write_text(model_yaml_text, encoding="utf-8")
print("Đã tạo:", MODEL_YAML)
print(model_yaml_text[:500], "...")

Đã tạo: /content/VR-TSD_YOLO/yolo11_uib_p2_traffic.yaml
nc: 59

backbone:
  - [-1, 1, UIBDown, [32, 3, 2, 6.0]]
  - [-1, 1, UIBDown, [64, 3, 2, 6.0]]
  - [-1, 1, UIB,     [64, 3, 1, 6.0]]
  - [-1, 1, UIB,     [64, 3, 1, 6.0]]

  - [-1, 1, UIBDown, [128, 3, 2, 6.0]]
  - [-1, 1, UIB,     [128, 3, 1, 6.0]]
  - [-1, 1, UIB,     [128, 3, 1, 6.0]]

  - [-1, 1, UIBDown, [256, 3, 2, 6.0]]
  - [-1, 1, UIB,     [256, 3, 1, 6.0]]
  - [-1, 1, UIB,     [256, 3, 1, 6.0]]

  - [-1, 1, UIBDown, [512, 3, 2, 6.0]]
  - [-1, 1, UIB,     [512, 3, 1, 6.0]]
  - [-1, 1, UIB ...


In [8]:
import torch

def suggest_train_cfg():
    if not torch.cuda.is_available():
        return {"device": "cpu", "imgsz": 640, "batch": 4, "workers": 2}

    mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    name = torch.cuda.get_device_name(0)
    print(f"GPU: {name} | VRAM ~ {mem_gb:.1f} GB")

    if mem_gb >= 35:
        return {"device": 0, "imgsz": 960, "batch": 24, "workers": 4}
    elif mem_gb >= 20:
        return {"device": 0, "imgsz": 960, "batch": 16, "workers": 4}
    elif mem_gb >= 14:
        return {"device": 0, "imgsz": 832, "batch": 12, "workers": 2}
    else:
        return {"device": 0, "imgsz": 768, "batch": 8, "workers": 2}

cfg = suggest_train_cfg()
DEVICE = cfg["device"]
IMGSZ = cfg["imgsz"]
BATCH = cfg["batch"]
WORKERS = cfg["workers"]

print("DEVICE  =", DEVICE)
print("IMGSZ   =", IMGSZ)
print("BATCH   =", BATCH)
print("WORKERS =", WORKERS)

GPU: NVIDIA A100-SXM4-80GB | VRAM ~ 79.3 GB
DEVICE  = 0
IMGSZ   = 960
BATCH   = 24
WORKERS = 4


In [9]:
import re
import inspect
import ultralytics.nn.tasks as tasks

# đăng ký thẳng vào namespace của tasks
tasks.UIB = UIB
tasks.UIBDown = UIBDown

def patch_parse_model_for_custom_modules():
    src = inspect.getsource(tasks.parse_model)

    # nếu đã patch rồi thì vẫn ép inject lại globals cho chắc
    if "base_modules = set(base_modules) | {UIB, UIBDown}" in src:
        tasks.parse_model.__globals__["UIB"] = UIB
        tasks.parse_model.__globals__["UIBDown"] = UIBDown
        tasks.__dict__["UIB"] = UIB
        tasks.__dict__["UIBDown"] = UIBDown
        print("parse_model đã patch từ trước, đã inject lại UIB/UIBDown")
        return

    lines = src.splitlines()
    out = []
    inserted = False
    in_base = False
    balance = 0
    indent_after = ""

    def delta_balance(line: str):
        return (line.count("{") - line.count("}")) + (line.count("(") - line.count(")"))

    for line in lines:
        out.append(line)

        if (not inserted) and (not in_base) and re.match(r"^\s*base_modules\s*=", line):
            in_base = True
            balance = delta_balance(line)
            indent_after = re.match(r"^(\s*)", line).group(1)

            if balance == 0:
                out.append(indent_after + "base_modules = set(base_modules) | {UIB, UIBDown}")
                out.append(indent_after + "base_modules = frozenset(base_modules)")
                inserted = True
                in_base = False

        elif in_base:
            balance += delta_balance(line)
            if balance == 0:
                out.append(indent_after + "base_modules = set(base_modules) | {UIB, UIBDown}")
                out.append(indent_after + "base_modules = frozenset(base_modules)")
                inserted = True
                in_base = False

    if not inserted:
        raise RuntimeError("Không patch được parse_model(): không tìm thấy base_modules")

    patched = "\n".join(out)

    g = dict(tasks.__dict__)
    g["UIB"] = UIB
    g["UIBDown"] = UIBDown

    exec(patched, g)
    tasks.parse_model = g["parse_model"]

    # inject lại globals của hàm mới tạo
    tasks.parse_model.__globals__["UIB"] = UIB
    tasks.parse_model.__globals__["UIBDown"] = UIBDown
    tasks.__dict__["UIB"] = UIB
    tasks.__dict__["UIBDown"] = UIBDown

    print("Đã patch parse_model() và inject UIB/UIBDown thành công")

patch_parse_model_for_custom_modules()

# kiểm tra thật sự đã có trong globals chưa
print("UIB in tasks.__dict__:", "UIB" in tasks.__dict__)
print("UIBDown in tasks.__dict__:", "UIBDown" in tasks.__dict__)
print("UIB in parse_model globals:", "UIB" in tasks.parse_model.__globals__)
print("UIBDown in parse_model globals:", "UIBDown" in tasks.parse_model.__globals__)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Đã patch parse_model() và inject UIB/UIBDown thành công
UIB in tasks.__dict__: True
UIBDown in tasks.__dict__: True
UIB in parse_model globals: True
UIBDown in parse_model globals: True


In [10]:
import os
import gc
import random
import shutil
import numpy as np
import torch
from ultralytics import YOLO

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.cuda.empty_cache()
gc.collect()

RUN_NAME = "vrtsd_uib_p2_fix2"
RUN_DIR = SAVE_ROOT / RUN_NAME

# nếu muốn chạy lại sạch hoàn toàn
if RUN_DIR.exists():
    shutil.rmtree(RUN_DIR)

IMGSZ = 832
BATCH = 8          # A100 40GB chạy mức này an toàn hơn nhiều
WORKERS = 2        # giảm worker để ổn định hơn

model = YOLO(str(MODEL_YAML))
model.load("yolo11n.pt")

results = model.train(
    data=str(DATA_YAML),

    epochs=200,
    imgsz=IMG_SZ if False else IMGSZ,  # giữ nguyên IMGSZ=832
    batch=BATCH,
    device=0,
    workers=WORKERS,

    optimizer="AdamW",
    lr0=1.0e-3,
    lrf=1.0e-2,
    weight_decay=5e-4,
    momentum=0.937,
    warmup_epochs=5.0,
    cos_lr=True,

    hsv_h=0.012,
    hsv_s=0.65,
    hsv_v=0.35,
    degrees=7.0,
    translate=0.10,
    scale=0.35,
    shear=2.0,
    perspective=0.0005,

    mosaic=1.0,
    mixup=0.05,
    close_mosaic=20,

    fliplr=0.0,
    flipud=0.0,
    multi_scale=False,   # QUAN TRỌNG: tắt để tránh ZeroDivisionError

    cache="disk",        # hoặc False nếu vẫn muốn an toàn hơn nữa
    amp=True,
    val=True,
    save=True,
    plots=True,
    patience=50,
    save_period=25,

    project=str(SAVE_ROOT),
    name=RUN_NAME,
    exist_ok=False,
    seed=SEED,
)

print("RUN_DIR =", RUN_DIR)
print("BEST    =", RUN_DIR / "weights" / "best.pt")
print("LAST    =", RUN_DIR / "weights" / "last.pt")

Transferred 2/1125 items from pretrained weights
Ultralytics 8.4.23 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=disk, cfg=None, classes=None, close_mosaic=20, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/VR-TSD_YOLO/data.yaml, degrees=7.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.012, hsv_s=0.65, hsv_v=0.35, imgsz=832, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.05, mode=train, model=/content/VR-TSD_YOLO/yolo11_uib_p2_traffic.yaml, momentum=0.937, mosaic=1.0, multi_scale=False, name=vrtsd_uib_p2_